# TritFlow：オープンソースPDK TR-1umで、三値のHDLからGDSへ

重みを−1・0・+1の三値に絞った大規模言語モデル（BitNet b1.58）では、行列積から乗算が消え、加算と減算だけが残る。三値のデータを三値のまま計算する回路は、改めて意味を持ち始めている。それでも、こうした計算を担っているのは二値の回路である。三値のまま論理を組む研究の多くは、CNTFETやメモリスタのような量産されていない素子か、シミュレーションの上にとどまる。三値のRTL合成の研究もあるが、その出力はゲートの回路網であり、レイアウトや製造までは扱わない。また、CMOS向けのオープンなツールでは三値の値を二値に変換して実装する。製造可能なオープンソースPDKの上で、三値のHDLからGDSまでを通す設計フローは、我々の知る限り報告されていない。

本ノートブックでは、この設計フローをTritFlowとして、日本で初めて商用ファブが公開したオープンソースPDK、OpenSUSI TR-1umの上で実現する。TritFlowは、著者がテープアウトした三値回路をスタンダードセルにし、三値のHDLから合成、配置配線、PDKのDRC/LVS、ポストレイアウトシミュレーションまでを一つのフローで行う。扱うのは平衡三値である。各桁は−1・0・+1のいずれかをとり、桁の重みは3のべき（1, 3, 9, …）である。回路の上では、それぞれ−5 V・0 V・+5 Vに対応する。

TritFlowのスタンダードセルの元になった三値回路は、日本初のOpenMPWサービスのシャトル（2026年、OpenSUSI-TR10）で、すでにテープアウトしている。このシャトルはオープンソースPDKとして公開されたTR-1umを使う。チップは2027年1月に届く。採択されれば、2月のISSCCでTritFlowを発表し、その土台となるセルの最初の実測結果を合わせて報告する。そしてTritFlowで設計した回路を、次のOpenSUSIシャトルで製造する。設計、製造、実測、報告を一巡させることで、TritFlowと、日本で芽生えたオープンソース半導体であるTR-1umが、Code-a-Chipの掲げる三つの価値を備えることを実証する。

**再現性**　後に続く研究者がTritFlowを中身まで理解し、自分で作り直し、説明できること。そのためにTritFlowは、セルの回路、論理合成のアルゴリズム、配置配線のいずれも、読んで追える単純さに保ちながら、PDKのDRC/LVSを通るテープアウト可能なGDSを出力するように設計した。本ノートブックは、各段がなぜそう動くのかを説明し、その場で動かして確かめられるようにしている。セルは実際に製造し実測する回路をもとにしているため、理解した内容をシリコンの実測と突き合わせることもできる（1〜6章）。

**再利用性**　そうして得た理解をもとに、TritFlowを独自に発展させられること。HDLを書き換えれば、自分の三値回路が同じ流れでGDSになり、MPWのフレームに載せたシャトル提出用の一式まで得られる。構成が単純なので、セルを足したり、合成や配置配線のアルゴリズムを置き換えたりすることもできる。セル、フロー、ノートブックはApache-2.0で公開しており、次の研究や次のシャトルの出発点になる（7・8章）。

**教育性**　この理解から発展までの道筋は、そのまま入門者の学びの道筋にもなる。すべてがノートブックの上で対話的に動き、各段で何が起きたかを図と数値で示すので、論理回路を学び始めた人でも、HDLを書いてからGDSができるまでを自分の手で一通りたどれる。しかもそれは、実際の製造と実測につながるフローである。

![fig_flow_v2](https://raw.githubusercontent.com/PenguinEino/tritflow/6ccf6973705488bb1653e163bcae3642be91e937/figures/out/fig_flow_v2_ja.png)

In [ ]:
#@title 準備（最初に実行） { display-mode: "form" }

import os, sys, json, time, subprocess, importlib.util, re
from pathlib import Path
SOURCE_URL = "https://github.com/PenguinEino/tritflow.git"
SOURCE_REF = os.environ.get("TRITFLOW_REF", "6ccf6973705488bb1653e163bcae3642be91e937")
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

def git(*args, progress=None):
    """Run git; with ``progress``, show git's own counter on one redrawn line."""
    if progress is None:
        return subprocess.check_output(["git", "-C", str(ROOT), *args], text=True).strip()
    from IPython.display import display, HTML
    style = "font:13px/1.6 ui-monospace,Menlo,Consolas,monospace;white-space:pre"
    line = display(HTML(f'<div style="{style}">⠿ {progress}</div>'), display_id=True)
    process, text, started = subprocess.Popen(["git", *args], stderr=subprocess.PIPE, text=True), "", time.monotonic()
    for chunk in iter(lambda: process.stderr.read(64), ""):
        text += chunk
        parts = [p.strip() for p in re.split(r"[\r\n]", text)[:-1] if p.strip()]
        if parts:
            line.update(HTML(f'<div style="{style}">⠿ <b>{progress}</b>  <span style="color:#8a8f98">{parts[-1][:80]}</span></div>'))
    if process.wait():
        raise RuntimeError(f"git {' '.join(args[:2])} failed:\n{text[-2000:]}")
    line.update(HTML(f'<div style="{style}"><span style="color:#1a9c4a">✓</span> <b>{progress}</b>  '
                     f'<span style="color:#8a8f98">{time.monotonic()-started:.1f}s</span></div>'))

if IN_COLAB:
    ROOT = Path("/content/tritflow")
    if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_REF):
        raise ValueError("TRITFLOW_REF must be a full 40-character commit")
    if not ROOT.exists():
        git("clone", "--progress", "--no-checkout", "--depth", "1", "--filter=blob:none",
            SOURCE_URL, str(ROOT), progress="GitHubからソースを取得")
        git("-C", str(ROOT), "fetch", "--progress", "--depth", "1", "origin", SOURCE_REF,
            progress=f"コミット {SOURCE_REF[:7]} を取得")
        git("-C", str(ROOT), "switch", "--detach", SOURCE_REF, progress="ファイルを展開")
    if git("rev-parse", "HEAD") != SOURCE_REF or git("status", "--porcelain", "--untracked-files=no"):
        # A runtime left by another version of this notebook: move it to the pinned commit and
        # drop the modules the kernel already imported from the old checkout.
        git("-C", str(ROOT), "fetch", "--progress", "--depth", "1", "origin", SOURCE_REF,
            progress=f"コミット {SOURCE_REF[:7]} に切り替え")
        git("reset", "--hard", SOURCE_REF)
        for name, module in list(sys.modules.items()):
            if str(getattr(module, "__file__", "") or "").startswith(str(ROOT)):
                del sys.modules[name]
else:
    candidates = [Path(os.environ.get("TRITFLOW_ROOT", ".")), Path.cwd(), Path.cwd().parent]
    ROOT = next((p.resolve() for p in candidates if (p/"tritflow").is_dir() and (p/"notebook").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("run from the TritFlow checkout or set TRITFLOW_ROOT")
sys.path[:0] = [str(ROOT), str(ROOT/"notebook")]
from live import step, run_stages
from runtime_support import SETUP_SCHEMA, setup_identity, check_runtime, print_tool_versions
runtime = ROOT/"notebook/runtime.json"
if IN_COLAB and (not runtime.is_file()
                 or json.loads(runtime.read_text()).get("TRITFLOW_SETUP_SHA256") != setup_identity(ROOT)):
    os.environ["TRITFLOW_NOTEBOOK_LANGUAGE"] = "ja"
    run_stages(["bash", str(ROOT/"notebook/colab_setup.sh"), "--source", str(ROOT)])
if runtime.is_file():
    os.environ.update(json.loads(runtime.read_text()))
    sys.path.insert(0, os.environ["TRITFLOW_PYTHON_TARGET"])
os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
with step("ツールを確認") as s:
    check_runtime()
    tools = print_tool_versions()
    s.done(" · ".join(re.search(r"(KLayout [\d.]+|ngspice-\d+)", v)[1] for v in tools.values()))
BUILD = Path(os.environ.get("TRITFLOW_NOTEBOOK_BUILD", "/content/build" if IN_COLAB else ROOT/"build/notebook")).resolve()

from live import need, provides, install_error_view, LayoutView
from views import pick_input, stimulus
from inverter_views import inverter
from cell_views import closure_view, cell_tables, cell_row
from tritflow import CELLS_DIR
from tritflow.sim.sizing import sweep_dimensions, load_grid
from tritflow import flow
from tritflow.viz.layout import gds_viewer
from tritflow.schematic import set_language
set_language("ja")
install_error_view()
for name, cell in {"source": "1-1 HDL", "net": "1-2 合成する", "lay": "1-3 配置配線する",
                   "lvs": "1-4 DRCとLVS", "post": "1-5 ポストレイアウトシミュレーション"}.items():
    provides(name, cell)


## 1. 最初の回路：`a & ~a` をGDSにする

本章では、小さな三値回路の1つである `a & ~a` を、HDLからGDSとポストレイアウトシミュレーションまで一通り通し、TritFlowのフロー全体を見る。各段の中身は、2章以降で順に開ける。

二値の論理では、`a & ~a` はaが何であっても偽になる。三値ではそうならない。

<img src="https://cdn.jsdelivr.net/gh/PenguinEino/tritflow@6ccf6973705488bb1653e163bcae3642be91e937/notebook/assets/tables/and.svg" width="696" alt="and">

TritFlowのHDL（TritVerilog）で書くと、次のようになる。

In [ ]:
# 1-1 HDL
source = """
module first(input trit a, output trit y);
  assign y = a & ~a;
endmodule
"""

論理合成する。三値の式を三値のセルに直接写像し、全入力の網羅で元のHDLとの等価性を検証する。

In [ ]:
# 1-2 合成する
need("source", why="合成するHDLは、1-1に書いてあります。")
with step("HDLを読み込んで合成") as s:
    net = flow.loads(source).synthesize()
    s.done(" · ".join(f"{cell}×{n}" for cell, n in net.report()["cells"].items())
           + (" · 全入力で等価" if net.equivalence["passed"] else " · 等価でない"))
net.schematic(net_names=True, expand="*", title="", legend=False)

配置配線する。三値セル用に作った配置配線器で、各セルを貫く3本の電源レール（VDD・VMID・VSS）を保ったまま、TR-1umのM1/M2の2層で配線する。

In [ ]:
# 1-3 配置配線する
need("net", why="配置配線する回路 net は、1-2で合成します。")
with step("配置配線", view=LayoutView()) as s:
    lay = net.place_and_route(output_dir=BUILD/net.top, seed=0)
    width, height = lay.stats["die_size_um"]
    s.done(f"{lay.stats['instance_count']}セル · {width:.0f} × {height:.0f} µm · 配線 {lay.stats['routed_nets']}本")

PDKの公式デッキでDRCとLVSを行う。

In [ ]:
# 1-4 DRCとLVS
need("lay", why="DRCとLVSは、1-3で作ったGDSを調べます。")
with step("DRC") as s:
    drc = lay.drc()
    s.done(f"違反 {drc['violations']}件")
with step("LVS") as s:
    lvs = lay.lvs()
    s.done("一致" if lvs["matched"] else "不一致")
gds_viewer(lay.gds_path)

ポストレイアウトシミュレーションを行う。GDSから見積もった配線容量を加え、各区間で出力が理想電圧（−5/0/+5 V）の±0.5 V以内に入るかを三値のテストベンチで判定する。ボタンで入力を選べる。

In [ ]:
# 1-5 ポストレイアウトシミュレーション
need("lvs", why="シミュレーションは、1-4でLVSが一致したGDSから回路を取り出して動かします。")
with step("抽出と配線容量") as s:
    extracted = lay.extract()
    signals = [n for n in extracted.capacitance["nets"] if n not in ("VDD", "VMID", "VSS")]
    s.done(f"信号{len(signals)}本に配線容量を追加")
with step("ngspice") as s:
    post = extracted.simulate(stimulus(net), load=100e-15, hold_s=1e-6, step_s=2e-9,
                              edge_s=1e-9, temperature_c=27)
    s.done("全区間で ±0.5 V の帯に入った" if post["measurements"]["passed"] else "帯に入らない区間がある")
pick_input(net, post)

ここまでのフローを振り返ると、次のようになる。

![fig_flow_map](https://raw.githubusercontent.com/PenguinEino/tritflow/6ccf6973705488bb1653e163bcae3642be91e937/figures/out/fig_flow_map_ja.png)

ここまでで、フロー全体を一通り体験した。次章からは、各部分の設計を詳しく見ていく。まず2章では、フローの土台であるセルのうち、最も基本的なINVの中を見る。

## 2. 三値のINV

1章の回路は、INVとNANDの2種類のセルでできていた。本章では、最も基本的なINVを取り出し、CMOSで3つの電圧レベルをどう作るかを見る。二値のINVとの違いは、0 Vを出すために上下のMOSを同時にオンにし、分圧で中間のレベルを作る点にある。

TritFlowのINVは、VDDとVSSの間にPMOS、抵抗、抵抗、NMOSを直列につないだ回路で、出力Yは2本の抵抗の間からとる。

<img src="https://cdn.jsdelivr.net/gh/PenguinEino/tritflow@6ccf6973705488bb1653e163bcae3642be91e937/notebook/assets/tables/inv.svg" width="302" alt="inv">

入力が±5 Vのときは片方のMOSだけがオンになり、Yはそちら側のレールに引かれる。入力が0 Vでは両方のMOSがオンになる。オンのMOSをオン抵抗 $r_{on} \propto (W/L)^{-1}$ で近似し、上側を $R_u = r_{on,P} + R_{RP}$、下側を $R_d = r_{on,N} + R_{RN}$ とすると、

$$V_Y = V_{SS} + (V_{DD}-V_{SS})\frac{R_d}{R_u+R_d},\qquad I = \frac{V_{DD}-V_{SS}}{R_u+R_d}$$

となる。$R_u = R_d$ のとき $V_Y$ は0 Vになり、その間は静的電流 $I$ が流れ続ける。寸法は、この釣り合いと電流の兼ね合いで決まる。

0 Vはどのレールからも供給されない。INVはVMIDに接続されておらず、0 Vは上下の釣り合いだけで作る。RRはオン抵抗の20倍以上あるので、0のレベルはMOSのばらつきではなく、同じ形の2本の抵抗の比で決まる。

<details><summary>テープアウトした回路図（Xschem）</summary>

![テープアウトした回路図（Xschem）](https://raw.githubusercontent.com/PenguinEino/tritflow/6ccf6973705488bb1653e163bcae3642be91e937/notebook/assets/schematics/INV.svg)

</details>

### SPICEで確かめ、寸法を変える

テープアウトした寸法のINVを、ngspiceとPDKのモデルでDC解析する。ほかの寸法は、テープアウトした寸法を含む90通りをSPICEで事前に計算した記録から選ぶ。記録は、回路、モデル、解析コードのハッシュで照合してから読む。ボタンで入力を、上のスライダーで寸法の候補を選ぶ。下の4本のスライダーは個々の寸法で、上のスライダーと連動する。

In [ ]:
# 2-1 INVのDC解析と寸法
with step("ngspice（INVのDC解析）") as s:
    spice = sweep_dimensions("INV", [], workdir=BUILD/"inv")
    zero = next(r for r in spice["rows"][0]["dc"] if r["input_trits"] == [0])
    s.done(f"入力 0 V：Y = {zero['output_v']['Y']:+.3f} V · {abs(zero['rail_current_a']['VDD'])*1e3:.3f} mA")
with step("寸法の記録を照合") as s:
    grid = load_grid("INV")
    s.done(f"{len(grid['rows'])}通り · 回路・モデル・解析コードが一致")
inverter(spice, grid)

## 3. セルの組：完全性とスタンダードセル

1章ではINVとNANDを使い、2章ではINVの中を見た。本章では、TritFlowのセルの組全体を見る。まず、なぜこの組で任意の三値論理を作れるのかを確かめ、次に、テープアウトした回路を配置配線できるスタンダードセルにする規則を見る。

### 完全性：この組で何でも作れるか

論理合成は、HDLで書いた論理を、用意したセルの組合せに置き換える作業である。どんな論理を書いても置き換えられるには、セルの組が**機能的に完全**である必要がある。機能的に完全とは、その組のセルをつなげば、あらゆる論理関数を作れることをいう。二値では、NANDが1種類あれば完全であることがよく知られている。NOTもANDもORも、NANDをつなげば作れるからである。

三値で同じことを確かめる。まず、入力が1本の論理関数（1入力関数）を考える。入力aは−1・0・+1の3通りで、それぞれに対して出力を3通りから選べるので、1入力関数は 3 × 3 × 3 = 27通りある。たとえばINVは、a = −1, 0, +1 をそれぞれ +1, 0, −1 に移す関数で、27通りのうちの1つである。

あるセルの組で、27通りのうちいくつを作れるかは、次の手順で数えられる。

1. はじめに手元にあるのは、入力aそのものと、定数セルが出す定数だけとする。
2. 手元の関数をセルの入力につなぎ、新しい関数を作る。
3. 作った関数を手元に加えて2に戻る。新しい関数が1つも増えなくなったら終わる。

最後に手元に残った関数が、その組で作れる1入力関数のすべてである。次のセルで、3つのセルの組について数える。

In [ ]:
# 3-1 1入力関数の閉包
from completeness import unary_closure, surjective_and_depends_on_both, inv, nand, nor, nany
bases = [("INV・NAND・NOR＋定数3つ", dict(unary=(inv,), binary=(nand, nor), constants=(-1, 0, 1))),
         ("NANY＋TIEZ", dict(binary=(nany,), constants=(0,))),
         ("NANY＋TIEP・TIEN", dict(binary=(nany,), constants=(-1, 1)))]
closures = [unary_closure(**basis) for _, basis in bases]
closure_view(bases, closures)
print("NANYは2つの入力の両方に依存し、出力に−1・0・+1のすべてが現れる：", surjective_and_depends_on_both(nany))

結果は、それぞれ次のように説明できる。

- **INV・NAND・NOR＋定数**：NANDとNORは、2つの入力の小さいほう（min）と大きいほう（max）の符号を反転したものである。0を「分からない」と読むと、min・max・符号反転は、分からない入力からは分からない出力しか作らない。このため、a = 0 での出力は、0になるか、aによらない定数になるかのどちらかに限られる。この条件を満たす関数しか作れない。
- **NANY＋TIEZ**：NANYは、2つの入力の和を−1から+1の範囲に切り詰め（sat）、符号を反転したセルで、$\mathrm{NANY}(a, b) = -\mathrm{sat}(a+b)$ である。入力を両方とも符号反転すると、出力も符号反転する。aと0から出発すると、作った関数はどれもこの対称性を持ち続けるので、$f(-a) = -f(a)$ を満たす関数しか作れない。
- **NANY＋TIEP・TIEN**：+1と−1の定数がこの対称性を崩し、27通りすべてに届く。

1入力関数がすべて作れるだけでは、まだ完全とは言えない。Słupecki（1939年）は、1入力関数をすべて作れ、さらに、2つの入力の両方に実際に依存し、出力に−1・0・+1のすべてが現れる2入力関数を1つ作れれば、入力が何本の三値関数でも作れることを示した。NANYはこの条件を満たすので、NANYとTIEP・TIENの組は機能的に完全である。TritFlowの論理合成は、この性質を使って、任意の真理値表をセルに置き換える（4章）。

### セルの組

TritFlowのセルの組を示す。INV・NAND・NOR・NANYはテープアウトした回路そのものである。BUFは1つの出力を多くの入力につなぐときの分割に、TIEP・TIEZ・TIENは定数に使う。

In [ ]:
# 3-2 セルの真理値表
cell_tables(CELLS_DIR/"out", ROOT/"notebook/assets/schematics")

### スタンダードセルにする

配置配線では、セルを横一列に並べ、隣どうしを突き当てて電源をつなぐ。そのため、すべてのセルが同じ高さと、同じ位置の電源レールを持つ必要がある。INV・NAND・NOR・NANYは、テープアウトした回路と素子寸法を保ったまま、この共通の枠に入れ直している。VDD・VMID・VSSの3本のレールがセルを横切り、信号ピンはM1の決まった高さに置いて、上を通るM2からビアで届くようにしている。左右反転して置いてもよい。

In [ ]:
# 3-3 スタンダードセル
import json
cell_row(json.loads((CELLS_DIR/"out/library.json").read_text()), CELLS_DIR/"out")